In [6]:
import os
import platform
import sqlite3
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime
import re
import csv

COLUNA_DATA = "FENTREGA"
COLUNA_ORIGEM = "ficheiro_origem"
NS = {"ss": "urn:schemas-microsoft-com:office:spreadsheet"}
SS_INDEX = "{urn:schemas-microsoft-com:office:spreadsheet}Index"
REGEX_ANO = re.compile(r"^(\d{4})")



In [7]:
if platform.system() == 'Windows':
    
    PASTA_FICHEIROS = Path(r"C:\Users\LISARR\Documents\python\01.Financeiro\weekly_input_csi")
    PASTA_DESTINO = Path(r"C:\Users\LISARR\Documents\python\01.Financeiro\Infor_27_weekly")
elif platform.system() == 'Darwin':
    
    PASTA_FICHEIROS = Path("/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/2026_dados")
    PASTA_DESTINO = Path("/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/Infor_27_weekly")
else:
    
    PASTA_FICHEIROS = Path("inform_27")



In [8]:
# Célula 1: Configuração de filtragem
ANO = 2026
MES = 8  # apenas 08
DIA_MAXIMO = 22

# Construir limites
MES_FILTRO = f"{ANO}{MES:02d}"
DATA_INICIO = MES_FILTRO + "01"
DATA_FIM = MES_FILTRO + str(DIA_MAXIMO).zfill(2)

In [9]:
# Célula 2: Função para extrair semana do ano
def obter_semana_ano(data_str):
    """Extrai semana do ano a partir de string YYYYMMDD."""
    try:
        data = datetime.strptime(str(data_str).strip(), "%Y%m%d")
        return data.isocalendar()[1]  # ISO semana (1-53)
    except:
        return None

In [10]:
# ==========================
# Converter XLS → CSV (Streaming, sem carregar tudo em RAM)
# ==========================
def ler_xls_streaming(caminho):
    """Retorna (cabecalho, generator de linhas)."""
    tree = ET.parse(caminho)
    raiz = tree.getroot()
    worksheets = raiz.findall(".//ss:Worksheet", NS)

    if not worksheets:
        raise ValueError("Nenhuma worksheet encontrada.")

    def ler_linha(row):
        valores = []
        proximo = 1
        for cell in row.findall("ss:Cell", NS):
            indice = cell.get(SS_INDEX)
            indice = int(indice) if indice else proximo
            while len(valores) < indice - 1:
                valores.append(None)
            data = cell.find("ss:Data", NS)
            valores.append(data.text if data is not None else None)
            proximo = indice + 1
        return valores

    # Extrair cabeçalho da 1ª sheet
    primeira_tabela = worksheets[0].find("ss:Table", NS)
    primeira_row = primeira_tabela.findall("ss:Row", NS)[0]
    cabecalho = ler_linha(primeira_row)
    cabecalho = [str(v).strip() if v else f"COLUNA_{i + 1}" for i, v in enumerate(cabecalho)]

    def gerar_linhas():
        for num_sheet, worksheet in enumerate(worksheets):
            tabela = worksheet.find("ss:Table", NS)
            if tabela is None:
                continue
            rows = tabela.findall("ss:Row", NS)
            if not rows:
                continue
            linhas_comeco = 1 if num_sheet == 0 else 0
            for row in rows[linhas_comeco:]:
                valores = ler_linha(row)
                if len(valores) < len(cabecalho):
                    valores += [None] * (len(cabecalho) - len(valores))
                yield valores[:len(cabecalho)]

    return cabecalho, gerar_linhas()


def filtrar_por_data(linha, cabecalho, data_inicio, data_fim):
    """Retorna True se a linha está no intervalo de datas."""
    try:
        indice_fentrega = cabecalho.index("FENTREGA")
        data_valor = str(linha[indice_fentrega]).strip()
        return data_inicio <= data_valor <= data_fim
    except (ValueError, IndexError):
        return True


def obter_semana_ano(data_str):
    """Extrai semana do ano a partir de string YYYYMMDD."""
    try:
        data = datetime.strptime(str(data_str).strip(), "%Y%m%d")
        return data.isocalendar()[1]
    except:
        return None


ficheiros = sorted(PASTA_FICHEIROS.rglob("*.xls"))

if not ficheiros:
    print(f"⚠️  Nenhum ficheiro .xls em: {PASTA_FICHEIROS}")
else:
    total_convertidos = 0

    for numero, caminho in enumerate(ficheiros, 1):
        try:
            contador = 0
            cabecalho, linhas_gen = ler_xls_streaming(caminho)
            
            cabecalho_com_semana = cabecalho + ["SEMANA_ANO"]

            with open(PASTA_DESTINO / "SAL_DAT027.csv", "w", newline="", encoding="utf-8", buffering=8192) as f:
                writer = csv.writer(f, delimiter=";")
                writer.writerow(cabecalho_com_semana)
                for linha in linhas_gen:
                    if filtrar_por_data(linha, cabecalho, DATA_INICIO, DATA_FIM):
                        try:
                            indice_fentrega = cabecalho.index("FENTREGA")
                            semana = obter_semana_ano(linha[indice_fentrega])
                        except:
                            semana = None
                        linha_com_semana = linha + [semana]
                        writer.writerow(linha_com_semana)
                        contador += 1

            total_convertidos += 1
            print(f"[{numero}/{len(ficheiros)}] {caminho.name} → CSV ({contador:,} linhas)")

        except Exception as erro:
            print(f"[{numero}/{len(ficheiros)}] {caminho.name} — ERRO: {erro}")

    print(f"\n✓ {total_convertidos} ficheiros convertidos")

[1/1] SAL_DAT027 (11).xls → CSV (64,310 linhas)

✓ 1 ficheiros convertidos
